# Capítulo 5 — Quickprop

Códigos do livro *Backpropagation e Redes Neurais* (Araken de Medeiros Santos).

> **Aviso:** este notebook foi gerado a partir dos scripts do livro e **não foi executado como notebook**; as saídas reportadas no texto vêm da execução dos scripts `.py` correspondentes. Execute as células na ordem, a partir desta pasta, com o ambiente de `requirements.txt` (ver Apêndice E). Valores podem diferir ligeiramente em outras plataformas.

In [ ]:
import os, sys
RAIZ = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, os.path.join(RAIZ, 'codigo'))
os.environ.setdefault('FIGS', os.path.join(RAIZ, 'figs'))
print('rnbook em', os.path.join(RAIZ, 'codigo'))

## Script `cap05/cap05.py`

In [ ]:
"""Cap. 5 - Quickprop: experimentos e figuras (semente fixa; números citados no texto)."""
import sys, os, time, warnings
warnings.filterwarnings("ignore")
import numpy as np
from rnbook import MLP
from rnbook import otimizadores as O
from rnbook.estilo import plt, COR, salvar
from rnbook.desenho import caixa, seta

In [ ]:
# ------------------------------------------------------------- 5.8 : quickprop 1D = secante na derivada
f = lambda w: np.exp(w) - 2 * w; s_ = lambda w: np.exp(w) - 2
w_prev = 0.0; eps = 0.3; dw = -eps * s_(w_prev); w = w_prev + dw      # 1º passo: gradiente
traj = [w_prev, w]
for _ in range(5):
    S, Sp = s_(traj[-1]), s_(traj[-2]); dwp = traj[-1] - traj[-2]
    traj.append(traj[-1] + S / (Sp - S) * dwp)
print("Quickprop 1D (f=e^w-2w, w0=0, eps=0.3):", [round(v, 6) for v in traj], "| ln2 =", round(np.log(2), 6))
sec = [traj[0], traj[1]]
for _ in range(5): sec.append(sec[-1] - s_(sec[-1]) * (sec[-1] - sec[-2]) / (s_(sec[-1]) - s_(sec[-2])))
print("máxima diferença entre Quickprop e secante em f':", max(abs(a - b) for a, b in zip(traj, sec)))
# quadrática exata: um passo
q = lambda w: 3 * (w - 2) ** 2 + 1; sq = lambda w: 6 * (w - 2)
w0, w1 = 0.0, 0.5; w2 = w1 + sq(w1) / (sq(w0) - sq(w1)) * (w1 - w0); print("quadrática 3(w-2)^2+1: w0=0,w1=0.5 -> w2 =", w2, "(mínimo em 2)")
# curvatura negativa: parábola para cima? (função cúbica com f''<0)
c = lambda w: -(w ** 3) / 3 + w; sc = lambda w: -(w ** 2) + 1
w0, w1 = 0.5, 0.8; w2 = w1 + sc(w1) / (sc(w0) - sc(w1)) * (w1 - w0)
print("curvatura negativa f=-w^3/3+w (f''<0 em w>0): w0=0.5, w1=0.8 -> passo quadrático leva a", round(w2, 4), "(f'' <0: isto é um MÁXIMO local em w=1, não um mínimo)")
fig, ax = plt.subplots(1, 2, figsize=(10.4, 3.7)); ww = np.linspace(-0.4, 1.6, 300)
ax[0].plot(ww, f(ww), color=COR["ink"], lw=2, label="$E(w)=e^w-2w$")
a0, a1 = traj[0], traj[1]; Sa0, Sa1 = s_(a0), s_(a1); curv = (Sa1 - Sa0) / (a1 - a0)
ax[0].plot(ww, f(a1) + Sa1 * (ww - a1) + 0.5 * curv * (ww - a1) ** 2, "--", color=COR["amb"], label="parábola local (curvatura da secante)")
ax[0].scatter(traj[:4], [f(v) for v in traj[:4]], color=COR["ind"], zorder=5); [ax[0].text(v, f(v) + 0.12, f"$w_{i}$", ha="center", fontsize=9, color=COR["indD"]) for i, v in enumerate(traj[:4])]
ax[0].set_ylim(0.4, 2.6); ax[0].legend(fontsize=8); ax[0].set_xlabel("w"); ax[0].set_title("Quickprop em uma variável")
ww2 = np.linspace(-0.2, 1.5, 300); ax[1].plot(ww2, sc(ww2), color=COR["ink"], lw=2, label="$E'(w)=1-w^2$ ($E=-w^3/3+w$)"); ax[1].axhline(0, color=COR["gry"], lw=0.8)
ax[1].plot([0.5, 0.8], [sc(0.5), sc(0.8)], "o--", color=COR["red"]); ax[1].scatter([w2], [0], color=COR["red"], marker="x", s=80, zorder=5)
ax[1].text(w2, 0.18, "passo quadrático", fontsize=8, color=COR["red"], ha="center"); ax[1].legend(fontsize=8, loc="lower left"); ax[1].set_xlabel("w"); ax[1].set_title("Falha: curvatura negativa → o passo vai a um máximo")
salvar(fig, "f5_parabola")

In [ ]:
# ------------------------------------------------------------- 5.5-5.7: gradientes explícitos x MLP.grad
def grad_explicito(W, b, V, c, X, T, phi, dphi, phio, dphio):
    """rede n-H-M: y_k = phio(sum_j V_kj phi(sum_i W_ji x_i + b_j) + c_k); E = 1/2 sum (y-t)^2. Laços explícitos."""
    N, n = X.shape; H = W.shape[0]; M = V.shape[0]
    gW, gb, gV, gc = np.zeros_like(W), np.zeros_like(b), np.zeros_like(V), np.zeros_like(c)
    for p in range(N):
        zj = np.array([W[j] @ X[p] + b[j] for j in range(H)]); aj = phi(zj)
        zk = np.array([V[k] @ aj + c[k] for k in range(M)]); yk = phio(zk)
        dk = (yk - T[p]) * dphio(zk)                       # delta da saída
        for k in range(M):
            gc[k] += dk[k]
            for j in range(H): gV[k, j] += dk[k] * aj[j]
        for j in range(H):
            dj = sum(dk[k] * V[k, j] for k in range(M)) * dphi(zj[j])        # delta oculto
            gb[j] += dj
            for i in range(n): gW[j, i] += dj * X[p, i]
    return gW, gb, gV, gc
rng = np.random.default_rng(1)
for (n, H, M, nome) in [(1, 4, 1, "5.5: 1 entrada, H ocultos, 1 saída"), (3, 5, 1, "5.6: n entradas, H ocultos, 1 saída"), (3, 5, 2, "5.7: n entradas, H ocultos, M saídas")]:
    X = rng.normal(size=(7, n)); T = rng.normal(size=(7, M)); r = MLP([n, H, M], ["tanh", "sigmoide"], semente=2)
    (Wm, bm), (Vm, cm) = r.desempacotar(r.theta)
    gW, gb, gV, gc = grad_explicito(Wm, bm, Vm, cm, X, T, np.tanh, lambda z: 1 - np.tanh(z) ** 2, lambda z: 1 / (1 + np.exp(-z)), lambda z: (1 / (1 + np.exp(-z))) * (1 - 1 / (1 + np.exp(-z))))
    g = r.grad(r.theta, X, T); ge = np.concatenate([gW.ravel(), gb, gV.ravel(), gc]); print(f"{nome}: diferença máx. explícito x MLP.grad = {np.abs(g-ge).max():.2e}")

In [ ]:
# ------------------------------------------------------------- GD x Quickprop em regressão (1-8-1, N=100)
rng = np.random.default_rng(3); N = 100; X = rng.uniform(0, 1, (N, 1)); T = np.sin(2 * np.pi * X) + rng.normal(0, 0.15, (N, 1))
def roda(otim, semente, **kw):
    r = MLP([1, 8, 1], ["tanh", "identidade"], semente=semente)
    fun = lambda th: r.perda(th, X, T) / N; gr = lambda th: r.grad(th, X, T) / N
    t0 = time.perf_counter(); res = otim(r.theta, fun, gr, **kw); return res, time.perf_counter() - t0
gradeG = [0.05, 0.1, 0.2, 0.5, 1.0, 2.0]
print("GD (lote), perda final E/N após 300 épocas, mediana de 10 sementes:")
resG = {}
for eta in gradeG:
    v = [roda(O.gradiente_descendente, s, taxa=eta, max_iter=300, tol_grad=0)[0].historico["perda"][-1] for s in range(10)]; resG[eta] = np.median(v); print("  eta=%.2f: %.4f" % (eta, np.median(v)))
etaG = 0.5   # maior taxa que não diverge em nenhuma das 10 sementes (eta=1.0 tem mediana menor, porém diverge em uma semente)
gradeQ = [0.05, 0.1, 0.2, 0.5, 1.0]
resQ = {}
print("Quickprop (mu=1.75), perda final após 300 épocas, mediana de 10 sementes:")
for eps_ in gradeQ:
    v = [roda(O.quickprop, s, epsilon=eps_, max_iter=300, tol_grad=0)[0].historico["perda"][-1] for s in range(10)]; resQ[eps_] = np.median(v); print("  epsilon=%.2f: %.4f (mín %.4f, máx %.4f)" % (eps_, np.median(v), np.min(v), np.max(v)))
epsQ = min(resQ, key=resQ.get); print("eta escolhido (GD) =", etaG, "| melhor epsilon (Quickprop) =", epsQ)
fig, ax = plt.subplots(1, 2, figsize=(10.4, 3.7)); hG, hQ = [], []
for s in range(10):
    hG.append(roda(O.gradiente_descendente, s, taxa=etaG, max_iter=300, tol_grad=0)[0].historico["perda"]); hQ.append(roda(O.quickprop, s, epsilon=epsQ, max_iter=300, tol_grad=0)[0].historico["perda"])
for h in hG: ax[0].semilogy(h, color=COR["amb"], alpha=0.45, lw=1)
for h in hQ: ax[0].semilogy(h, color=COR["ind"], alpha=0.45, lw=1)
ax[0].semilogy([], [], color=COR["amb"], label=f"gradiente descendente ($\\eta$={etaG})"); ax[0].semilogy([], [], color=COR["ind"], label=f"Quickprop ($\\epsilon$={epsQ}, $\\mu$=1,75)"); ax[0].axhline(0.5 * 0.15 ** 2, color=COR["gry"], ls=":"); ax[0].legend(fontsize=8)
ax[0].set_xlabel("época"); ax[0].set_ylabel("perda $E/N$"); ax[0].set_title("10 sementes, rede 1–8–1, lote")
finG = [h[-1] for h in hG]; finQ = [h[-1] for h in hQ]
print("após 300 épocas: GD mediana %.4f [%.4f, %.4f] | Quickprop mediana %.4f [%.4f, %.4f] | piso do ruído %.5f" % (np.median(finG), min(finG), max(finG), np.median(finQ), min(finQ), max(finQ), 0.5 * 0.15 ** 2))
for lim in (0.03, 0.02):
    eG = [int(np.argmax(np.array(h) < lim)) if (np.array(h) < lim).any() else None for h in hG]; eQ = [int(np.argmax(np.array(h) < lim)) if (np.array(h) < lim).any() else None for h in hQ]
    print("épocas para E/N<%.2f: GD %s | Quickprop %s" % (lim, eG, eQ))
ax[1].boxplot([finG, finQ], tick_labels=["gradiente\ndescendente", "Quickprop"], showfliers=True); ax[1].set_yscale("log"); ax[1].set_ylabel("perda final $E/N$ (300 épocas)"); ax[1].set_title("Dispersão entre 10 sementes")
salvar(fig, "f5_quickprop_gd")

In [ ]:
# ------------------------------------------------------------- sensibilidade a mu e epsilon
mus = [1.0, 1.25, 1.5, 1.75, 2.0, 3.0, 5.0]; fig, ax = plt.subplots(figsize=(6.4, 3.8)); tab = {}
for eps_ in (0.05, 0.2, 0.5):
    med = []; 
    for mu in mus:
        v = []
        for s in range(10):
            res = roda(O.quickprop, s, epsilon=eps_, mu=mu, max_iter=300, tol_grad=0)[0]; ff = res.historico["perda"][-1]; v.append(ff if np.isfinite(ff) else np.inf)
        med.append(np.median(v))
    tab[eps_] = med; ax.semilogy(mus, med, "o-", label=f"$\\epsilon$={eps_}")
ax.set_xlabel("fator máximo de crescimento $\\mu$"); ax.set_ylabel("perda final mediana $E/N$"); ax.legend(); ax.set_title("Quickprop: sensibilidade a $\\mu$ e $\\epsilon$ (300 épocas)"); salvar(fig, "f5_sensibilidade")
for eps_, med in tab.items(): print("epsilon=%.2f:" % eps_, {mu: round(float(m), 4) for mu, m in zip(mus, med)})

In [ ]:
# ------------------------------------------------------------- deslocamento 0,1 da derivada (Fahlman) no XOR
P4 = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], float); T4 = np.array([[0], [1], [1], [0]], float)
def xor_taxa(ativ_saida, otim, H=2, **kw):
    ok = 0; ep = []
    for s in range(30):
        r = MLP([2, H, 1], ["sigmoide", ativ_saida], semente=s, init="pequena")
        res = otim(r.theta, lambda th: r.perda(th, P4, T4), lambda th: r.grad(th, P4, T4), tol_perda=1e-3, **kw)
        acerto = ((r.prever(P4, res.theta) > 0.5).astype(float) == T4).all(); ok += int(acerto)
        if acerto: ep.append(res.n_iter)
    return ok, (int(np.median(ep)) if ep else None)
print("XOR (2-2-1, 30 sementes), até 3000 épocas, parada em E<1e-3:")
print("  GD (eta=1)       sigmoide:", xor_taxa("sigmoide", O.gradiente_descendente, taxa=1.0, max_iter=3000))
print("  Quickprop(eps=.5) sigmoide:", xor_taxa("sigmoide", O.quickprop, epsilon=0.5, max_iter=3000))
print("  Quickprop(eps=.5) sigmoide_fahlman (+0.1):", xor_taxa("sigmoide_fahlman", O.quickprop, epsilon=0.5, max_iter=3000))
print("  GD (eta=1) sigmoide_fahlman (+0.1):", xor_taxa("sigmoide_fahlman", O.gradiente_descendente, taxa=1.0, max_iter=3000))
rF = MLP([2, 2, 1], ["sigmoide", "sigmoide_fahlman"], semente=0); from rnbook import grad_numerico, erro_relativo
print("gradcheck com o artifício +0.1: erro relativo =", "%.2e" % erro_relativo(rF.grad(rF.theta, P4, T4), grad_numerico(lambda th: rF.perda(th, P4, T4), rF.theta)), "(esperado: grande, pois não é o gradiente verdadeiro)")

In [ ]:
# ------------------------------------------------------------- denominador pequeno/zero: tratamento
print("denominador zero: s=s_ant com mesmo sinal:", end=" ")
th = np.array([1.0]); out = O.quickprop(th, lambda t: float(0.5 * np.sum(t ** 2)), lambda t: np.array([0.0]) + 0 * t, max_iter=3)
print("rodou sem erro; theta final", out.theta)
print("OK cap05")

## Exercícios práticos e projeto: soluções de referência

### 1

Implemente a função `passo_quickprop(S, S_ant, dw_ant, eps, mu)` com as regras (A) a (D) (para os dois sentidos do passo anterior) e use-a para minimizar $E(w)=e^w-2w$ a partir de $w_0=0$.

1. Teste os três casos do Exemplo 5.1 ($\Delta w(t-1)=-0{,}2$, $S_{\text{ant}}=0{,}60$ e $S\in\{0{,}30;\,0{,}50;\,-0{,}20\}$).
2. Minimize $E$ com $\varepsilon=0{,}3$ e $\mu=1{,}75$, imprimindo $w$, $E'(w)$ e a regra usada a cada iteração, até $|E'|<10^{-8}$.
3. Compare o número de iterações com o do gradiente descendente com $\eta=0{,}3$ e com o de Newton (Seção 2.8).

**Pergunta:** por que o Quickprop se parece com o método das secantes aplicado a $E'$?

In [ ]:
import numpy as np

def passo_quickprop(S, S_ant, dw_ant, eps=0.1, mu=1.75):
    sigma = mu / (1.0 + mu)
    if dw_ant == 0:
        return -eps * S, "A"
    sinal = -1.0 if dw_ant < 0 else 1.0
    passo = -eps * S if sinal * S < 0 else 0.0
    if sinal * S <= sinal * sigma * S_ant:
        return passo + mu * dw_ant, "C"
    return passo + dw_ant * S / (S_ant - S), "D"

for S in (0.30, 0.50, -0.20):
    print("caso S=%+.2f: passo = %+.4f (regra %s)" % ((S,) + passo_quickprop(S, 0.60, -0.2)))

dE = lambda w: np.exp(w) - 2.0
w, dw, S_ant = 0.0, 0.0, 0.0
for k in range(1, 40):
    S = dE(w)
    if abs(S) < 1e-8:
        break
    passo, regra = passo_quickprop(S, S_ant, dw, eps=0.3, mu=1.75)
    print("k=%2d  w=%.6f  E'=%+.2e  passo=%+.5f  regra %s" % (k, w, S, passo, regra))
    S_ant, dw, w = S, passo, w + passo
print("Quickprop: %d iterações, w = %.8f (ln 2 = %.8f)" % (k, w, np.log(2)))

x, k_gd = 0.0, 0
while abs(dE(x)) >= 1e-8:
    x -= 0.3 * dE(x); k_gd += 1
print("gradiente descendente (eta=0,3): %d iterações" % k_gd)
x, k_n = 0.0, 0
while abs(dE(x)) >= 1e-8:
    x -= dE(x) / np.exp(x); k_n += 1
print("Newton: %d iterações" % k_n)

### 2

Use a base sintética **Friedman #1** (`make_friedman1`, Friedman, 1991): $y=10\sin(\pi x_1x_2)+20(x_3-0{,}5)^2+10x_4+5x_5+\varepsilon$, com 10 atributos (só 5 relevantes). Compare, com a **mesma arquitetura** (10–8–1, tanh), o gradiente descendente em lote e o Quickprop (pacote `rnbook`) com `MLPRegressor` (`lbfgs`, `adam`, `sgd`) do scikit-learn.

1. Gere 400 amostras (`noise=1.0`, `random_state=0`), separe 25% para teste e padronize $X$ e $y$ **com o treino**.
2. Rode 5 sementes de cada método, com 300 épocas (iterações): gradiente descendente em lote $\eta=0{,}5$; Quickprop $\varepsilon=0{,}2$ e $\mu=1{,}75$; scikit-learn com `max_iter=300` (para `sgd`, `learning_rate_init=0.05`; para `adam`, o padrão).
3. Registre a mediana (e o intervalo) do $R^2$ de teste e o tempo.

**Pergunta:** o que torna a comparação "injusta" ou apenas "não equivalente"?

In [ ]:
import time
import warnings
import numpy as np
from sklearn.datasets import make_friedman1
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from rnbook import MLP, otimizadores as O

warnings.filterwarnings("ignore")
X, y = make_friedman1(n_samples=400, n_features=10, noise=1.0, random_state=0)
X_a, X_te, y_a, y_te = train_test_split(X, y, test_size=0.25, random_state=0)
esx, esy = StandardScaler().fit(X_a), StandardScaler().fit(y_a[:, None])
Xs, Xt, ys = esx.transform(X_a), esx.transform(X_te), esy.transform(y_a[:, None])
N = len(ys)

def propria(otim, semente, **kw):
    r = MLP([10, 8, 1], ["tanh", "identidade"], semente=semente)
    t0 = time.perf_counter()
    res = otim(r.theta, lambda th: r.perda(th, Xs, ys) / N, lambda th: r.grad(th, Xs, ys) / N, **kw)
    pred = esy.inverse_transform(r.prever(Xt, res.theta)).ravel()
    return r2_score(y_te, pred), time.perf_counter() - t0

def sk(solver, semente, **kw):
    t0 = time.perf_counter()
    m = MLPRegressor(hidden_layer_sizes=(8,), activation="tanh", solver=solver, alpha=0.0,
                     max_iter=300, random_state=semente, **kw).fit(Xs, ys.ravel())
    pred = esy.inverse_transform(m.predict(Xt)[:, None]).ravel()
    return r2_score(y_te, pred), time.perf_counter() - t0

metodos = {
    "GD em lote (eta=0.5)": lambda s: propria(O.gradiente_descendente, s, taxa=0.5, max_iter=300, tol_grad=0),
    "Quickprop (eps=0.2)": lambda s: propria(O.quickprop, s, epsilon=0.2, max_iter=300, tol_grad=0),
    "sklearn lbfgs": lambda s: sk("lbfgs", s),
    "sklearn adam (padrão)": lambda s: sk("adam", s),
    "sklearn sgd (lr 0.05)": lambda s: sk("sgd", s, learning_rate_init=0.05),
}
for nome, f in metodos.items():
    R = [f(s) for s in range(5)]
    r2 = [r[0] for r in R]
    print("%-24s R2 teste: mediana %.3f [%.3f, %.3f] | tempo médio %.2f s"
          % (nome, np.median(r2), min(r2), max(r2), np.mean([r[1] for r in R])))

### 3

Na mesma base **Friedman #1**, estude a sensibilidade do Quickprop.

1. Varie $\varepsilon\in\{0{,}02;\,0{,}05;\,0{,}2;\,0{,}5\}$ e $\mu\in\{1{,}25;\,1{,}75;\,3\}$ (5 sementes, 300 épocas).
2. Para cada par, registre a mediana do $R^2$ de teste e o número de sementes com $R^2<0{,}5$.
3. Descreva o padrão.

**Pergunta:** qual dos dois parâmetros importa mais? Por quê?

In [ ]:
import warnings
import numpy as np
from sklearn.datasets import make_friedman1
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from rnbook import MLP, otimizadores as O

warnings.filterwarnings("ignore")
X, y = make_friedman1(n_samples=400, n_features=10, noise=1.0, random_state=0)
X_a, X_te, y_a, y_te = train_test_split(X, y, test_size=0.25, random_state=0)
esx, esy = StandardScaler().fit(X_a), StandardScaler().fit(y_a[:, None])
Xs, Xt, ys = esx.transform(X_a), esx.transform(X_te), esy.transform(y_a[:, None])
N = len(ys)

def r2_teste(eps, mu, semente):
    r = MLP([10, 8, 1], ["tanh", "identidade"], semente=semente)
    res = O.quickprop(r.theta, lambda th: r.perda(th, Xs, ys) / N, lambda th: r.grad(th, Xs, ys) / N,
                      epsilon=eps, mu=mu, max_iter=300, tol_grad=0)
    return r2_score(y_te, esy.inverse_transform(r.prever(Xt, res.theta)).ravel())

print("   eps \\ mu " + "".join("%14.2f" % m for m in (1.25, 1.75, 3.0)))
for eps in (0.02, 0.05, 0.2, 0.5):
    linha = []
    for mu in (1.25, 1.75, 3.0):
        r2 = [r2_teste(eps, mu, s) for s in range(5)]
        linha.append("%7.3f (%d)" % (np.median(r2), sum(v < 0.5 for v in r2)))
    print("%10.2f   " % eps + "   ".join(linha))

### Orçamento igual: Quickprop, gradiente descendente e L-BFGS em dois problemas

**Objetivo.** Comparar de forma controlada o Quickprop com o gradiente descendente em lote e com o `lbfgs` do scikit-learn, em regressão e em classificação.

**Dados.** (a) Regressão: Friedman #1 (`make_friedman1`, 400 amostras). (b) Classificação: **Wine** (UCI, 178 amostras, 13 atributos, 3 classes), com alvos *one-hot* e saída sigmoide (erro quadrático, pacote `rnbook`), em comparação com `MLPClassifier`.

**Tarefas.**
1. Em (a): 10–8–1; em (b): 13–8–3 (tanh nas ocultas e sigmoide na saída para o pacote `rnbook`). Separe teste (25%, estratificado em (b)); padronize com o treino.
2. Mesmo orçamento: 300 épocas; 5 sementes. Gradiente descendente com a melhor taxa de uma pequena grade; Quickprop com a melhor $\varepsilon$ de uma pequena grade (escolha pela perda **de treino**; o teste só para relatar).
3. Relate: $R^2$ (a) ou acurácia (b) de teste (mediana e intervalo), perda final de treino e tempo.
4. Redija 10 a 12 linhas sobre o que é e o que não é comparável (épocas de `lbfgs` $\neq$ épocas do lote, perdas diferentes) e sobre a variabilidade.

**Para ir além (não testado no livro).** Repita com bases de regressão e classificação do repositório da UCI e do Kaggle (por exemplo, resistência do concreto ou qualidade de vinhos). Verifique licença, tipos de colunas e valores ausentes antes de usar.

In [ ]:
import time
import warnings
import numpy as np
from sklearn.datasets import load_wine, make_friedman1
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.preprocessing import StandardScaler
from rnbook import MLP, otimizadores as O

warnings.filterwarnings("ignore")

def avaliar(nome, f):
    R = [f(s) for s in range(5)]
    q = [r[0] for r in R]
    print("%-26s teste: mediana %.3f [%.3f, %.3f] | perda treino (mediana) %s | tempo %.2f s"
          % (nome, np.median(q), min(q), max(q),
             ("%.4f" % np.median([r[1] for r in R])) if R[0][1] is not None else "  --  ", np.mean([r[2] for r in R])))

def melhor_param(grade, f_treino):
    return min(grade, key=lambda g: np.median([f_treino(g, s) for s in range(5)]))

# ---------------- (a) regressão
X, y = make_friedman1(n_samples=400, n_features=10, noise=1.0, random_state=0)
X_a, X_te, y_a, y_te = train_test_split(X, y, test_size=0.25, random_state=0)
esx, esy = StandardScaler().fit(X_a), StandardScaler().fit(y_a[:, None])
Xs, Xt, ys = esx.transform(X_a), esx.transform(X_te), esy.transform(y_a[:, None]); N = len(ys)

def reg(otim, s, **kw):
    r = MLP([10, 8, 1], ["tanh", "identidade"], semente=s); t0 = time.perf_counter()
    res = otim(r.theta, lambda th: r.perda(th, Xs, ys) / N, lambda th: r.grad(th, Xs, ys) / N, max_iter=300, tol_grad=0, **kw)
    return r2_score(y_te, esy.inverse_transform(r.prever(Xt, res.theta)).ravel()), res.historico["perda"][-1], time.perf_counter() - t0

eta = melhor_param([0.1, 0.2, 0.5, 1.0], lambda g, s: reg(O.gradiente_descendente, s, taxa=g)[1])
eps = melhor_param([0.02, 0.05, 0.2, 0.5], lambda g, s: reg(O.quickprop, s, epsilon=g)[1])
print("(a) Friedman #1 | eta=%.2f, eps=%.2f (escolhidos pela perda de treino)" % (eta, eps))
avaliar("GD em lote", lambda s: reg(O.gradiente_descendente, s, taxa=eta))
avaliar("Quickprop", lambda s: reg(O.quickprop, s, epsilon=eps))
def reg_sk(s):
    t0 = time.perf_counter()
    m = MLPRegressor(hidden_layer_sizes=(8,), activation="tanh", solver="lbfgs", alpha=0.0, max_iter=300, random_state=s).fit(Xs, ys.ravel())
    return r2_score(y_te, esy.inverse_transform(m.predict(Xt)[:, None]).ravel()), None, time.perf_counter() - t0
avaliar("sklearn lbfgs", reg_sk)

# ---------------- (b) classificação (Wine)
Xw, yw = load_wine(return_X_y=True)
Xa, Xte, ya, yte = train_test_split(Xw, yw, test_size=0.25, random_state=0, stratify=yw)
esc = StandardScaler().fit(Xa); Xs2, Xt2 = esc.transform(Xa), esc.transform(Xte); T = np.eye(3)[ya]; N2 = len(ya)

def cls(otim, s, **kw):
    r = MLP([13, 8, 3], ["tanh", "sigmoide"], semente=s); t0 = time.perf_counter()
    res = otim(r.theta, lambda th: r.perda(th, Xs2, T) / N2, lambda th: r.grad(th, Xs2, T) / N2, max_iter=300, tol_grad=0, **kw)
    return np.mean(np.argmax(r.prever(Xt2, res.theta), axis=1) == yte), res.historico["perda"][-1], time.perf_counter() - t0

eta = melhor_param([0.1, 0.2, 0.5, 1.0], lambda g, s: cls(O.gradiente_descendente, s, taxa=g)[1])
eps = melhor_param([0.02, 0.05, 0.2, 0.5], lambda g, s: cls(O.quickprop, s, epsilon=g)[1])
print("(b) Wine | eta=%.2f, eps=%.2f (escolhidos pela perda de treino)" % (eta, eps))
avaliar("GD em lote", lambda s: cls(O.gradiente_descendente, s, taxa=eta))
avaliar("Quickprop", lambda s: cls(O.quickprop, s, epsilon=eps))
def cls_sk(s):
    t0 = time.perf_counter()
    m = MLPClassifier(hidden_layer_sizes=(8,), activation="tanh", solver="lbfgs", alpha=0.0, max_iter=300, random_state=s).fit(Xs2, ya)
    return m.score(Xt2, yte), None, time.perf_counter() - t0
avaliar("sklearn lbfgs", cls_sk)